In [1]:
# Run this cell if libraries are not installed
# !pip install scikit-learn pandas numpy

In [2]:
import json
import os
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime

print('Libraries imported successfully!')

Libraries imported successfully!


In [3]:
class TaskManager:
    def __init__(self, filename='tasks.json'):
        self.filename = filename
        self.tasks = []
        self.load_tasks()

    # ── File I/O ──────────────────────────────────────────────
    def load_tasks(self):
        """Load tasks from JSON file."""
        if os.path.exists(self.filename):
            with open(self.filename, 'r') as f:
                self.tasks = json.load(f)
            print(f'Loaded {len(self.tasks)} tasks from {self.filename}')
        else:
            self.tasks = []
            print('No saved tasks found. Starting fresh!')

    def save_tasks(self):
        """Save tasks to JSON file."""
        with open(self.filename, 'w') as f:
            json.dump(self.tasks, f, indent=4)
        print(f'Tasks saved to {self.filename}')

    # ── CRUD Operations ───────────────────────────────────────
    def add_task(self, title, description='', priority='Medium', category='General'):
        """Add a new task."""
        task = {
            'id': len(self.tasks) + 1,
            'title': title,
            'description': description,
            'priority': priority,           # High / Medium / Low
            'category': category,
            'status': 'Pending',
            'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }
        self.tasks.append(task)
        self.save_tasks()
        print(f'✅ Task added: "{title}" [Priority: {priority}]')
        return task

    def remove_task(self, task_id):
        """Remove a task by ID."""
        for task in self.tasks:
            if task['id'] == task_id:
                self.tasks.remove(task)
                self.save_tasks()
                print(f'🗑️ Task {task_id} removed: "{task["title"]}"')
                return
        print(f'Task ID {task_id} not found!')

    def list_tasks(self, filter_status=None, filter_priority=None):
        """List all tasks with optional filters."""
        tasks = self.tasks
        if filter_status:
            tasks = [t for t in tasks if t['status'] == filter_status]
        if filter_priority:
            tasks = [t for t in tasks if t['priority'] == filter_priority]

        if not tasks:
            print('No tasks found.')
            return

        df = pd.DataFrame(tasks)[['id', 'title', 'priority', 'category', 'status', 'created_at']]
        print(f'\n📋 Task List ({len(tasks)} tasks):')
        print(df.to_string(index=False))
        return df

    def complete_task(self, task_id):
        """Mark a task as completed."""
        for task in self.tasks:
            if task['id'] == task_id:
                task['status'] = 'Completed'
                self.save_tasks()
                print(f'✔️ Task {task_id} marked as Completed!')
                return
        print(f'Task ID {task_id} not found!')

    def prioritize_tasks(self):
        """Sort tasks by priority: High > Medium > Low."""
        priority_order = {'High': 1, 'Medium': 2, 'Low': 3}
        sorted_tasks = sorted(
            [t for t in self.tasks if t['status'] == 'Pending'],
            key=lambda x: priority_order.get(x['priority'], 4)
        )
        print('\n🔢 Prioritized Pending Tasks:')
        df = pd.DataFrame(sorted_tasks)[['id', 'title', 'priority', 'category']]
        print(df.to_string(index=False))
        return sorted_tasks

    # ── ML Recommendation ─────────────────────────────────────
    def recommend_tasks(self, query, top_n=3):
        """
        ML-based task recommendation using TF-IDF + Cosine Similarity.
        Given a query description, recommends the most relevant pending tasks.
        """
        pending = [t for t in self.tasks if t['status'] == 'Pending']
        if len(pending) < 2:
            print('Not enough pending tasks to recommend. Add more tasks first!')
            return

        # Combine title + description for each task
        corpus = [f"{t['title']} {t['description']} {t['category']}" for t in pending]
        corpus.append(query)   # Add user query at the end

        # TF-IDF Vectorization
        vectorizer = TfidfVectorizer(stop_words='english')
        tfidf_matrix = vectorizer.fit_transform(corpus)

        # Cosine similarity between query and all tasks
        query_vector = tfidf_matrix[-1]          # last row = query
        task_vectors = tfidf_matrix[:-1]          # all others = tasks
        similarities = cosine_similarity(query_vector, task_vectors).flatten()

        # Get top N most similar tasks
        top_indices = similarities.argsort()[::-1][:top_n]

        print(f'\n🤖 ML Recommendations for: "{query}"')
        print('-' * 50)
        for rank, idx in enumerate(top_indices, 1):
            t = pending[idx]
            score = similarities[idx]
            print(f'{rank}. [{t["priority"]}] {t["title"]} (Score: {score:.3f})')
            if t['description']:
                print(f'   → {t["description"]}')

print('TaskManager class defined!')

TaskManager class defined!


In [4]:
tm = TaskManager()

Loaded 14 tasks from tasks.json


In [5]:
# Add a variety of tasks
tm.add_task('Fix login bug', 'Resolve authentication error on mobile', priority='High', category='Development')
tm.add_task('Write project report', 'Summarize Q2 ML model performance', priority='High', category='Documentation')
tm.add_task('Team meeting prep', 'Prepare slides for weekly standup', priority='Medium', category='Management')
tm.add_task('Update README', 'Add setup instructions to GitHub repo', priority='Low', category='Development')
tm.add_task('Data preprocessing', 'Clean and normalize the dataset for training', priority='High', category='ML')
tm.add_task('Code review', 'Review PRs submitted by the team', priority='Medium', category='Development')
tm.add_task('Model evaluation', 'Evaluate XGBoost model on test set', priority='High', category='ML')
tm.add_task('Buy groceries', 'Milk, eggs, bread', priority='Low', category='Personal')

Tasks saved to tasks.json
✅ Task added: "Fix login bug" [Priority: High]
Tasks saved to tasks.json
✅ Task added: "Write project report" [Priority: High]
Tasks saved to tasks.json
✅ Task added: "Team meeting prep" [Priority: Medium]
Tasks saved to tasks.json
✅ Task added: "Update README" [Priority: Low]
Tasks saved to tasks.json
✅ Task added: "Data preprocessing" [Priority: High]
Tasks saved to tasks.json
✅ Task added: "Code review" [Priority: Medium]
Tasks saved to tasks.json
✅ Task added: "Model evaluation" [Priority: High]
Tasks saved to tasks.json
✅ Task added: "Buy groceries" [Priority: Low]


{'id': 22,
 'title': 'Buy groceries',
 'description': 'Milk, eggs, bread',
 'priority': 'Low',
 'category': 'Personal',
 'status': 'Pending',
 'created_at': '2026-05-27 19:19:40'}

In [6]:
tm.list_tasks()


📋 Task List (22 tasks):
 id                title priority      category    status          created_at
  1        Fix login bug     High   Development Completed 2026-05-18 13:52:11
  2 Write project report     High Documentation   Pending 2026-05-18 13:52:11
  3    Team meeting prep   Medium    Management   Pending 2026-05-18 13:52:11
  4        Update README      Low   Development   Pending 2026-05-18 13:52:11
  5   Data preprocessing     High            ML   Pending 2026-05-18 13:52:11
  6          Code review   Medium   Development   Pending 2026-05-18 13:52:11
  7     Model evaluation     High            ML   Pending 2026-05-18 13:52:11
  9 Write project report     High Documentation   Pending 2026-05-26 18:33:34
 10    Team meeting prep   Medium    Management   Pending 2026-05-26 18:33:34
 11        Update README      Low   Development   Pending 2026-05-26 18:33:34
 12   Data preprocessing     High            ML   Pending 2026-05-26 18:33:34
 13          Code review   Medium   Dev

,id,title,priority,category,status,created_at
0,1,Fix login bug,High,Development,Completed,2026-05-18 13:52:11
1,2,Write project report,High,Documentation,Pending,2026-05-18 13:52:11
2,3,Team meeting prep,Medium,Management,Pending,2026-05-18 13:52:11
3,4,Update README,Low,Development,Pending,2026-05-18 13:52:11
4,5,Data preprocessing,High,ML,Pending,2026-05-18 13:52:11
5,6,Code review,Medium,Development,Pending,2026-05-18 13:52:11
6,7,Model evaluation,High,ML,Pending,2026-05-18 13:52:11
7,9,Write project report,High,Documentation,Pending,2026-05-26 18:33:34
8,10,Team meeting prep,Medium,Management,Pending,2026-05-26 18:33:34
9,11,Update README,Low,Development,Pending,2026-05-26 18:33:34


In [7]:
tm.prioritize_tasks()


🔢 Prioritized Pending Tasks:
 id                title priority      category
  2 Write project report     High Documentation
  5   Data preprocessing     High            ML
  7     Model evaluation     High            ML
  9 Write project report     High Documentation
 12   Data preprocessing     High            ML
 14     Model evaluation     High            ML
 15        Fix login bug     High   Development
 16 Write project report     High Documentation
 19   Data preprocessing     High            ML
 21     Model evaluation     High            ML
  3    Team meeting prep   Medium    Management
  6          Code review   Medium   Development
 10    Team meeting prep   Medium    Management
 13          Code review   Medium   Development
 17    Team meeting prep   Medium    Management
 20          Code review   Medium   Development
  4        Update README      Low   Development
 11        Update README      Low   Development
 15        Buy groceries      Low      Personal
 18       

[{'id': 2,
  'title': 'Write project report',
  'description': 'Summarize Q2 ML model performance',
  'priority': 'High',
  'category': 'Documentation',
  'status': 'Pending',
  'created_at': '2026-05-18 13:52:11'},
 {'id': 5,
  'title': 'Data preprocessing',
  'description': 'Clean and normalize the dataset for training',
  'priority': 'High',
  'category': 'ML',
  'status': 'Pending',
  'created_at': '2026-05-18 13:52:11'},
 {'id': 7,
  'title': 'Model evaluation',
  'description': 'Evaluate XGBoost model on test set',
  'priority': 'High',
  'category': 'ML',
  'status': 'Pending',
  'created_at': '2026-05-18 13:52:11'},
 {'id': 9,
  'title': 'Write project report',
  'description': 'Summarize Q2 ML model performance',
  'priority': 'High',
  'category': 'Documentation',
  'status': 'Pending',
  'created_at': '2026-05-26 18:33:34'},
 {'id': 12,
  'title': 'Data preprocessing',
  'description': 'Clean and normalize the dataset for training',
  'priority': 'High',
  'category': 'ML',


In [8]:
# Recommend tasks related to machine learning work
tm.recommend_tasks('machine learning model training and evaluation')


🤖 ML Recommendations for: "machine learning model training and evaluation"
--------------------------------------------------
1. [High] Model evaluation (Score: 0.308)
   → Evaluate XGBoost model on test set
2. [High] Model evaluation (Score: 0.308)
   → Evaluate XGBoost model on test set
3. [High] Model evaluation (Score: 0.308)
   → Evaluate XGBoost model on test set


In [9]:
# Recommend tasks related to coding/development
tm.recommend_tasks('software development and code fixes')


🤖 ML Recommendations for: "software development and code fixes"
--------------------------------------------------
1. [Medium] Code review (Score: 0.224)
   → Review PRs submitted by the team
2. [Medium] Code review (Score: 0.224)
   → Review PRs submitted by the team
3. [Medium] Code review (Score: 0.224)
   → Review PRs submitted by the team


In [10]:
tm.complete_task(1)  # Mark Task ID 1 as done

Tasks saved to tasks.json
✔️ Task 1 marked as Completed!


In [11]:
tm.remove_task(8)  # Remove Task ID 8

Task ID 8 not found!


In [12]:
tm.list_tasks(filter_status='Pending', filter_priority='High')


📋 Task List (10 tasks):
 id                title priority      category  status          created_at
  2 Write project report     High Documentation Pending 2026-05-18 13:52:11
  5   Data preprocessing     High            ML Pending 2026-05-18 13:52:11
  7     Model evaluation     High            ML Pending 2026-05-18 13:52:11
  9 Write project report     High Documentation Pending 2026-05-26 18:33:34
 12   Data preprocessing     High            ML Pending 2026-05-26 18:33:34
 14     Model evaluation     High            ML Pending 2026-05-26 18:33:34
 15        Fix login bug     High   Development Pending 2026-05-27 19:19:40
 16 Write project report     High Documentation Pending 2026-05-27 19:19:40
 19   Data preprocessing     High            ML Pending 2026-05-27 19:19:40
 21     Model evaluation     High            ML Pending 2026-05-27 19:19:40


,id,title,priority,category,status,created_at
0,2,Write project report,High,Documentation,Pending,2026-05-18 13:52:11
1,5,Data preprocessing,High,ML,Pending,2026-05-18 13:52:11
2,7,Model evaluation,High,ML,Pending,2026-05-18 13:52:11
3,9,Write project report,High,Documentation,Pending,2026-05-26 18:33:34
4,12,Data preprocessing,High,ML,Pending,2026-05-26 18:33:34
5,14,Model evaluation,High,ML,Pending,2026-05-26 18:33:34
6,15,Fix login bug,High,Development,Pending,2026-05-27 19:19:40
7,16,Write project report,High,Documentation,Pending,2026-05-27 19:19:40
8,19,Data preprocessing,High,ML,Pending,2026-05-27 19:19:40
9,21,Model evaluation,High,ML,Pending,2026-05-27 19:19:40
